In [1]:
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from src.utils import generate_mask_tensor
from src.embedding import embed
from src.gp_ccm import GP_ccm_sig, run_sigGPCCM_experiment
from src.sp_ccm import run_SP_CCM, SP_CCM_iaaft, run_ccm_experiment
from src.iaaft import surrogates

from scipy.stats import ranksums
torch.set_printoptions(sci_mode = False)

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



In [77]:
co2_norm = torch.load("data/CO2_vostok_stan_400kyr_timeseries.pt").to(torch.float32)

# Initalise
g = torch.tensor([0.4, 0.3, 0.2])

for t in range(2, co2_norm.shape[0] + 1):
    
    # Co2 is external forcing 
    g_next = (g[t] * (1.9 - (1.9 * g[t]) - (0.4 * co2_norm[t - 2])))

    g = torch.concat((g, g_next.unsqueeze(0)))

g_norm = g.sub(g.mean(dim = -1).unsqueeze(-1)).div(g.std(dim = -1).unsqueeze(-1))[0:401]

In [78]:
co2_norm = torch.load("data/CO2_vostok_stan_400kyr_timeseries.pt").to(torch.float32)

# Initalise
# g = torch.tensor([0.4, 0.3, 0.2])
g = torch.tensor([0.2, 0.1, 0.1])

for t in range(2, co2_norm.shape[0] + 1):
    
    # Co2 is external forcing 
    g_next = (g[t] * (1.4 - (1.9 * g[t]) - (0.2 * co2_norm[t - 2])))

    g = torch.concat((g, g_next.unsqueeze(0)))

g_norm = g.sub(g.mean(dim = -1).unsqueeze(-1)).div(g.std(dim = -1).unsqueeze(-1))[0:401]

In [79]:
MAX = 400

fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, co2_norm.shape[0])[0:MAX], y = co2_norm[0:MAX],
                    mode = 'lines',
                    name = 'F',
                    line_color = "red"))

fig.add_trace(go.Scatter(x = torch.arange(0, co2_norm.shape[0])[0:MAX], y = g_norm[0:MAX],
                    mode = 'lines',
                    name = 'G',
                    line_color = "purple"))

fig.update_layout(title = 'Noisy time series',
                   xaxis_title = 't',
                   yaxis_title = 'values')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")
fig.update_layout(xaxis_range=[-2, MAX])

fig.update_layout(autosize = False, width = 1000, height = 400)

fig.show()

In [80]:
# GLOBALS
k = 3
N_TRAIN = torch.tensor([100]).to(device)

##############
### GP-CCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device) + 2 # k -1 

NOISE_SCALE = torch.tensor([0.05], device = device) # for diagonal
RBF_SCALE = torch.tensor([0.25], device = device)

############
### ECCM ###
############

ccm_filter = torch.ones(size = (k, )).to(device) # same as sig filter
ccm_shift = torch.tensor(ccm_filter.shape[0] - 1).to(device) + 2

# CO2 -> G

In [73]:
### sig-GP_CCM ###
CO2G_gpccm_rho_mean, CO2G_gpccm_rho_sd, CO2G_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = co2_norm.to(device),
    causal_y = g_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.434
Rho std 0.103
Rho indep. p95 0.283


In [74]:
### CCM ###
CO2G_ccm_rho_mean, CO2G_ccm_rho_sd, CO2G_ccm_rho_ind_p95, CO2G_ccm_noise =  run_ccm_experiment(
    causal_x = co2_norm.to(device),
    causal_y = g_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.977
Rho std 0.009
Rho indep. p95 0.583
Added noise 0.0


# G -> CO2

In [81]:
### sig-GP_CCM ###
GCO2_gpccm_rho_mean, GCO2_gpccm_rho_sd, GCO2_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = g_norm.to(device),
    causal_y = co2_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

Rho mean 0.22
Rho std 0.06
Rho indep. p95 0.255


In [82]:
### CCM ###
GCO2_ccm_rho_mean, GCO2_ccm_rho_sd, GCO2_ccm_rho_ind_p95, GCO2_ccm_noise =  run_ccm_experiment(
    causal_x = g_norm.to(device),
    causal_y = co2_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

We get nan's and have to increase the noise level.
Rho mean 0.569
Rho std 0.154
Rho indep. p95 0.487
Added noise 0.025
